# Interviewa Colab Setup

This notebook is designed to run in **Google Colab with a T4 GPU runtime**.

## What this notebook does
- Installs the runtime dependencies.
- Starts Ollama locally inside Colab.
- Pulls the required LLM and embedding models.
- Starts the Flask AI service on port `5001`.
- Creates a public ngrok URL and prints the API URL you need for your backend `.env` file.

## Before you run
1. Open this notebook in Google Colab.
2. Go to **Runtime → Change runtime type**.
3. Select **Hardware accelerator: GPU** and choose **T4** if available.
4. Run the cells **top to bottom**.
5. When the notebook prints `AI_SERVICE_URL=...`, copy that value into your backend `.env` file as `AI_SERVICE_URL`.

## Important
- If the Colab runtime disconnects or restarts, the ngrok URL will change.
- Re-run the notebook and copy the new `AI_SERVICE_URL` into `.env` again.
- Keep the backend running locally after updating `.env`.


In [ ]:
# Cell 1 — Install everything
!pip install -q ollama flask pyngrok openai-whisper
# Replace TTS with edge-tts
!pip install -q edge-tts

# Install Ollama manually (Colab is Linux)
# Fix zstd first, then install ollama
!apt-get install -y zstd
!curl -fsSL https://ollama.ai/install.sh | sh

In [ ]:
import subprocess
import time

# Start Ollama server in background
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("Ollama server started")

# Pull models (this takes 5-10 mins depending on connection)
print("Pulling llama3.1:8b ...")
!ollama pull llama3.1:8b

print("Pulling nomic-embed-text ...")
!ollama pull nomic-embed-text

print("Pulling deepseek-coder:6.7b ...")
!ollama pull deepseek-coder:6.7b

print("All models ready!")

In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.1:8b",
        "prompt": "Say hello in one sentence.",
        "stream": False
    }
)
print(response.json()["response"])

In [ ]:
import requests

r = requests.post("http://localhost:11434/api/generate",
    json={"model": "llama3.1:8b", "prompt": "say hi", "stream": False})
print(r.status_code)
print(r.text[:200])

In [ ]:
from flask import Flask, request, jsonify, send_file, Response
import whisper, edge_tts, asyncio, tempfile, os, threading, time
import requests as req

app = Flask(__name__)
stt_model = whisper.load_model("small")
VOICE = "en-US-GuyNeural"
OLLAMA = "http://localhost:11434"

@app.route("/health")
def health():
    return jsonify({"status": "ok"})

# Proxy ALL ollama routes transparently
@app.route("/api/generate", methods=["POST"])
def generate():
    r = req.post(f"{OLLAMA}/api/generate", json=request.json)
    return jsonify(r.json())

@app.route("/api/embeddings", methods=["POST"])
def embeddings():
    r = req.post(f"{OLLAMA}/api/embeddings", json=request.json)
    return jsonify(r.json())

@app.route("/api/tags", methods=["GET"])
def tags():
    r = req.get(f"{OLLAMA}/api/tags")
    return jsonify(r.json())

# REPLACE IT WITH:
@app.route("/transcribe", methods=["POST"])
def transcribe():
    audio = request.files["audio"]
    with tempfile.NamedTemporaryFile(delete=False, suffix=".webm") as f:
        audio.save(f.name)
        result = stt_model.transcribe(
            f.name,
            language="en",
            temperature=0.0,
            initial_prompt="Technical software engineering mock interview. Keywords: React, SQL, API, Git, database, loop, function, variable, docker, algorithm, system design, frontend, backend."
        )
        os.unlink(f.name)
    return jsonify({"text": result["text"]})

@app.route("/speak", methods=["POST"])
def speak():
    text = request.json.get("text", "")
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    tmp.close()
    async def generate_audio():
        communicate = edge_tts.Communicate(text, VOICE)
        await communicate.save(tmp.name)
    asyncio.run(generate_audio())
    return send_file(tmp.name, mimetype="audio/mpeg")

threading.Thread(
    target=lambda: app.run(port=5001, use_reloader=False),
    daemon=True
).start()
time.sleep(3)
print("Server running on port 5001")

## Get the API URL for your backend

Run the next cell after the Flask service starts. It will print a public `AI_SERVICE_URL` from ngrok.

Copy that URL exactly and paste it into your backend `.env` file:

```env
AI_SERVICE_URL=https://your-ngrok-url.ngrok-free.app
```

If the Colab session restarts, you must run the ngrok cell again and update `.env` with the new URL.


In [ ]:

# Cell 4 — ngrok + Ollama tunnel
from pyngrok import ngrok, conf

NGROK_TOKEN = "use your ngrok token here"
conf.get_default().auth_token = NGROK_TOKEN

tunnel = ngrok.connect(5001, "http")
AI_SERVICE_URL = tunnel.public_url

print("\n========= COPY INTO YOUR LOCAL .env =========")
print(f"AI_SERVICE_URL={AI_SERVICE_URL}")
print("==============================================\n")

In [ ]:
import requests

BASE = AI_SERVICE_URL

# Health
print("Health:", requests.get(f"{BASE}/health").json())

# LLM
r = requests.post(f"{BASE}/api/generate",
    json={"model": "llama3.1:8b", "prompt": "Say hi in 5 words.", "stream": False})
print("LLM status:", r.status_code)
print("LLM response:", r.json().get("response"))

# TTS
r = requests.post(f"{BASE}/speak",
    json={"text": "Hello, I am your AI interviewer today."})
print("TTS status:", r.status_code, "| bytes:", len(r.content))